In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoTokenizer

from llminfer.models.qwen3 import Qwen3ForCausalLM
from llminfer.kv_cache import KVCache

/Users/ericchen/Eric/llm-infer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = Qwen3ForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

In [7]:
input_ids = torch.tensor([[0, 1, 2]])
position_ids = torch.arange(0, input_ids.shape[1]).unsqueeze(0)
print("input_ids: ", input_ids, "position_ids: ", position_ids)

input_ids:  tensor([[0, 1, 2]]) position_ids:  tensor([[0, 1, 2]])


In [8]:
model.eval()
with torch.no_grad():
    output = model(input_ids)

# output.logits[0, -2:, :]

q:  tensor([[ 0.2806, -0.1316,  0.1063,  ...,  1.5951,  1.0073, -1.9601],
        [-0.6900, -1.5319,  0.6965,  ..., -0.0335,  2.3896, -1.1887],
        [-0.8980,  3.5425, -1.6077,  ...,  0.5218, -0.4913, -0.6499],
        ...,
        [ 0.3748,  4.3628,  1.2547,  ..., -2.4573,  1.1548,  0.1684],
        [ 1.6655,  2.0235,  0.9187,  ..., -0.3946, -0.1357, -1.6459],
        [ 1.9662, -0.2537,  0.5032,  ..., -0.1757, -0.5996, -0.5430]])
k:  torch.Size([1, 8, 3, 128])
v:  torch.Size([1, 8, 3, 128])
attn_out:  tensor([[-0.0395, -0.0768, -0.0097,  ..., -0.0123, -0.0718,  0.0294],
        [-0.0442, -0.0806, -0.0110,  ..., -0.0148, -0.0744,  0.0314],
        [ 0.1809, -0.0877,  0.2086,  ...,  0.0583, -0.1149,  0.1025],
        ...,
        [ 0.0134, -0.0506, -0.0234,  ...,  0.0064, -0.0366,  0.0126],
        [-0.0122,  0.0550,  0.0900,  ...,  0.0735, -0.1296, -0.0078],
        [ 0.0002,  0.0799,  0.0184,  ...,  0.0544, -0.0584,  0.0279]])
out:  tensor([ 0.2307, -0.2497, -0.0516,  ..., -0.0006,

In [10]:
# with kv cache
kv_cache = KVCache(
    batch_size=1,
    num_layers=model.config.num_hidden_layers,
    max_seq_len=512,
    num_heads=model.config.num_key_value_heads,
    head_dim=model.config.head_dim,
)
kv_cache.reset()

with torch.no_grad():
    # prefill
    prev_output = model(input_ids[:, :-1], position_ids=position_ids[:, :-1], kv_cache=kv_cache)

    kv_cache_input_ids = input_ids[:, -1:]
    kv_cache_position_ids = position_ids[:, -1:]
    # output = model(kv_cache_input_ids, position_ids=kv_cache_input_ids)
    output = model(kv_cache_input_ids, position_ids=kv_cache_input_ids, kv_cache=kv_cache)

# output.logits[:, -1, :]

q:  tensor([[ 0.2806, -0.1316,  0.1063,  ...,  1.5951,  1.0073, -1.9601],
        [-0.6900, -1.5319,  0.6965,  ..., -0.0335,  2.3896, -1.1887],
        [-0.8980,  3.5425, -1.6077,  ...,  0.5218, -0.4913, -0.6499],
        ...,
        [ 0.3748,  4.3627,  1.2547,  ..., -2.4573,  1.1548,  0.1684],
        [ 1.6655,  2.0235,  0.9187,  ..., -0.3946, -0.1357, -1.6459],
        [ 1.9662, -0.2537,  0.5032,  ..., -0.1757, -0.5996, -0.5430]])
k:  torch.Size([1, 8, 3, 128])
v:  torch.Size([1, 8, 3, 128])
attn_out:  tensor([[-0.0395, -0.0768, -0.0097,  ..., -0.0123, -0.0718,  0.0294],
        [-0.0442, -0.0806, -0.0110,  ..., -0.0148, -0.0744,  0.0314],
        [ 0.1809, -0.0877,  0.2086,  ...,  0.0583, -0.1149,  0.1025],
        ...,
        [ 0.0134, -0.0506, -0.0234,  ...,  0.0064, -0.0366,  0.0126],
        [-0.0122,  0.0550,  0.0900,  ...,  0.0735, -0.1296, -0.0078],
        [ 0.0002,  0.0799,  0.0184,  ...,  0.0544, -0.0584,  0.0279]])
out:  tensor([ 0.2307, -0.2497, -0.0516,  ..., -0.0006,

In [7]:
prev_output.logits[:, -1:, :]

tensor([[[ 6.9337,  3.9790,  6.5550,  ..., -0.8974, -0.8974, -0.8974]]])